<a href="https://colab.research.google.com/github/diazcas2/mIArmario/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# 1. Primero NumPy fijado
!pip install "numpy==1.26.4" --quiet

# 2. PyTorch y visión
!pip install torch torchvision --quiet

# 3. Transformers y Hugging Face
!pip install transformers huggingface_hub qwen-vl-utils --quiet

# 4. rembg (traerá Pillow>=12 automáticamente)
!pip install "rembg[cpu]" onnxruntime --quiet

# 5. Google GenAI (solo el nuevo)
!pip install google-genai --quiet

# 6. Resto de utilidades
!pip install google-search-results langdetect beautifulsoup4 gradio_client --quiet

# 7. Fuentes
!apt-get install -y fonts-dejavu -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 192.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 18.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but

In [ ]:
import json
import re
import time
import torch
import google.generativeai as genai
from transformers import pipeline #Si da error al importarlo reinicia el entorno y no vuelvas a ejecutar el chunk anterior
from google.colab import userdata
from serpapi import GoogleSearch
import requests
import os


from transformers import WhisperProcessor, WhisperForConditionalGeneration
from IPython.display import display, Image as IPImage

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/MIARMARIO/')

from Modulo1 import *
from Modulo3 import *
from Modulo2 import *
from Modulo4 import *


In [ ]:
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
whisper = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3",torch_dtype=torch.float32 )
device = "cuda" if torch.cuda.is_available() else "cpu"
whisper_model = whisper.to(device)

In [ ]:
GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')   # Google Cloud → Custom Search API
SEARCH_ENGINE_ID = userdata.get('SEARCH_ENGINE_ID') # programmablesearchengine.google.com → cx

SERPAPI_KEY = userdata.get('SERPAPI_KEY')  # serpapi.com → dashboard → API Key

CARPETA_ROPA      = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO"          # Carpeta con fotos del armario
RUTA_ARMARIO_JSON = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/armario.json"  # BD persistente de prendas

CARPETA_SALIDA    = '/content/drive/MyDrive/Colab Notebooks/MIARMARIO/CARPETA SALIDA'

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

# Paso 1. Cargar armario.

In [ ]:
'''if forzar_reconstruccion and os.path.exists(RUTA_ARMARIO_JSON):
        os.remove(RUTA_ARMARIO_JSON)
        print("[Paso 1] Armario eliminado para reconstrucción completa")'''

if os.path.exists(RUTA_ARMARIO_JSON):
    print("[Paso 1] Usando armario.json existente.")
    armario = cargar_armario(RUTA_ARMARIO_JSON)
elif os.path.exists(CARPETA_ROPA):
    print("[Paso 1] Construyendo armario desde las fotos...")
    armario = modulo2_construir_armario(
        carpeta_ropa=CARPETA_ROPA,
        ruta_armario_json=RUTA_ARMARIO_JSON,
    )
else:
    raise FileNotFoundError("No hay armario.json ni carpeta ARMARIO. Añade fotos para continuar.")


In [ ]:
print(json.dumps(armario, ensure_ascii=False, indent=2))

# Paso 2. Procesar instrucciones del ususario.


In [ ]:
resultado_mod1 = modulo1_entrada(
    fuente="texto",
    texto="hola quiero que me digas un outfit que sea bueno para dar un paseo por la noche",
    ciudad="Granada",
    model=gemini_model
)



In [ ]:
# Ejemplo 2 — Entrada por voz
resultado_mod1 = modulo1_entrada(
    fuente="voz",
    ruta_audio="/content/drive/MyDrive/Colab Notebooks/MIARMARIO/mi_audio.mp4",# <-- cambia por tu archivo
    ciudad="Sevilla",
    model=model,
    whisper=whisper,
    processor=processor,
    device=device
)



In [ ]:
print(json.dumps(resultado_mod1, ensure_ascii=False, indent=2))

# Paso 3. Seleccionar outfit.

In [ ]:
resultado_mod3 = modulo3_seleccion_outfit(
    armario_json=armario,
    instruccion_usuario=resultado_mod1["texto"],
    contexto=resultado_mod1["contexto"],
    model=gemini_model,
    SERPAPI_KEY=SERPAPI_KEY
)



In [ ]:
print(json.dumps(resultado_mod3, ensure_ascii=False, indent=2))

In [ ]:
ruta_jpg,ruta_lookbook = modulo4_generar_imagen(
    resultado_mod3 = resultado_mod3,
    armario        = armario,
    carpeta_salida = CARPETA_SALIDA,
    model          = gemini_model,
)

#display(IPImage(filename=ruta_lookbook))
print(f'Lookbook guardado en: {ruta_lookbook}')


# Mostrar el HTML directamente en el notebook
from IPython.display import HTML, display

with open(ruta_lookbook, "r", encoding="utf-8") as f:
    contenido_html = f.read()

display(HTML(contenido_html))

In [ ]:
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
whisper = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3",torch_dtype=torch.float32 )
device = "cuda" if torch.cuda.is_available() else "cpu"
whisper_model = whisper.to(device)

In [ ]:
print("Cargando Qwen2-VL...")
qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
qwen_model = (
    Qwen2VLForConditionalGeneration
    .from_pretrained("Qwen/Qwen2-VL-2B-Instruct", torch_dtype=torch.float16)
    .eval()
    .to(device)
)
print("Qwen2-VL listo ✓")

# BACKEND FLASK


In [ ]:
!pip install flask flask-cors pyngrok -q

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading, time, os, tempfile, base64, traceback, sys

GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')   # Google Cloud → Custom Search API
SEARCH_ENGINE_ID = userdata.get('SEARCH_ENGINE_ID') # programmablesearchengine.google.com → cx

SERPAPI_KEY = userdata.get('SERPAPI_KEY')  # serpapi.com → dashboard → API Key

CARPETA_ROPA      = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO"          # Carpeta con fotos del armario
RUTA_ARMARIO_JSON = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/armario.json"  # BD persistente de prendas

CARPETA_SALIDA    = '/content/drive/MyDrive/Colab Notebooks/MIARMARIO/CARPETA SALIDA'

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.5-flash")



# ─────────────────────────────────────────────
# LOG A FICHERO (visible con: !tail -f /content/debug.log)
# ─────────────────────────────────────────────
LOG_PATH = "/content/debug.log"

def log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")

# Limpiar log anterior
with open(LOG_PATH, "w") as f:
    f.write("=== NUEVO ARRANQUE ===\n")

# ─────────────────────────────────────────────
app = Flask(__name__)
CORS(app, resources={r"/*": {"origins": "*"}},
     allow_headers=["Content-Type", "Authorization", "ngrok-skip-browser-warning"],
     methods=["GET", "POST", "OPTIONS"])


NGROK_TOKEN    = userdata.get('TOKEN_NGROK')
CARPETA_SALIDA = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/lookbooks"
RUTA_ARMARIO   = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/armario.json"
CARPETA_ROPA   = "/content/drive/MyDrive/Colab Notebooks/MIARMARIO/ARMARIO"

os.makedirs(CARPETA_SALIDA, exist_ok=True)

# ─────────────────────────────────────────────
# CORS
# ─────────────────────────────────────────────
@app.after_request
def add_cors_headers(response):
    response.headers["Access-Control-Allow-Origin"] = "*"
    response.headers["Access-Control-Allow-Headers"] = "Content-Type,Authorization,ngrok-skip-browser-warning"
    response.headers["Access-Control-Allow-Methods"] = "GET,POST,OPTIONS"
    return response


@app.route("/armario", methods=["POST", "OPTIONS"])
def subir_armario():
    if request.method == "OPTIONS":
        return jsonify({"ok": True}), 200

    try:
        fotos = request.files.getlist("fotos")
        if not fotos:
            return jsonify({"error": "No se recibieron imágenes"}), 400

        guardadas = []
        for foto in fotos:
            if foto and foto.filename:
                ruta = os.path.join(CARPETA_ROPA, foto.filename)
                foto.save(ruta)
                guardadas.append(foto.filename)
                log(f"[armario] Guardada: {foto.filename}")

        if not guardadas:
            return jsonify({"error": "No se pudo guardar ninguna imagen"}), 400

        log(f"[armario] Reconstruyendo armario...")
        armario = modulo2_construir_armario(
            carpeta_ropa=CARPETA_ROPA,
            ruta_armario_json=RUTA_ARMARIO,
            model=qwen_model,
            processor=qwen_processor
        )
        log(f"[armario] OK → {len(armario)} prendas")
        return jsonify({
            "mensaje": f"{len(guardadas)} prenda(s) añadidas · {len(armario)} prendas en total",
            "guardadas": guardadas,
            "total": len(armario)
        })

    except Exception as e:
        log(f"[armario] ERROR: {e}")
        log(traceback.format_exc())
        return jsonify({"error": str(e)}), 500  # ← siempre JSON, nunca HTML


# ─────────────────────────────────────────────
# PING
# ─────────────────────────────────────────────
@app.route("/ping", methods=["GET"])
def ping():
    return jsonify({"status": "ok"})

# ─────────────────────────────────────────────
# LÓGICA PRINCIPAL — corre en el hilo principal
# para que Gemini no se bloquee
# ─────────────────────────────────────────────
_resultado_pendiente = {}
_lock = threading.Lock()

def _procesar_outfit(job_id, instruccion, ciudad, ruta_audio, fotos_paths):
    """Corre en el hilo principal de Python donde Gemini funciona correctamente."""
    try:
        log(f"[{job_id}] Iniciando procesamiento")

        # ── Módulo 1 ──────────────────────────────────
        log(f"[{job_id}] M1 — entrada")
        if ruta_audio:
            log(f"[{job_id}] M1 — transcribiendo voz")
            resultado_m1 = modulo1_entrada(
                fuente="voz",
                ruta_audio=ruta_audio,
                ciudad=ciudad,
                model=gemini_model,
                whisper=whisper,
                processor=processor,
                device=device
            )
            os.unlink(ruta_audio)
        else:
            log(f"[{job_id}] M1 — normalizando texto: {instruccion[:50]}")
            resultado_m1 = modulo1_entrada(
                fuente="texto",
                texto=instruccion,
                ciudad=ciudad,
                model=gemini_model
            )
        log(f"[{job_id}] M1 OK → {resultado_m1.get('texto','')[:60]}")

        # ── Módulo 2 ──────────────────────────────────
        log(f"[{job_id}] M2 — armario")
        if fotos_paths:
            armario = modulo2_construir_armario(
                carpeta_ropa=CARPETA_ROPA,
                ruta_armario_json=RUTA_ARMARIO,
                model=qwen_model,
                processor=qwen_processor
            )
        else:
            if not os.path.exists(RUTA_ARMARIO):
                raise FileNotFoundError("No existe armario.json y no se subieron fotos")
            armario = cargar_armario(RUTA_ARMARIO)
        log(f"[{job_id}] M2 OK → {len(armario)} prendas")

        # ── Módulo 3 ──────────────────────────────────
        log(f"[{job_id}] M3 — seleccionando outfit")
        resultado_m3 = modulo3_seleccion_outfit(
            armario_json=armario,
            instruccion_usuario=resultado_m1["texto"],
            contexto=resultado_m1["contexto"],
            model=gemini_model,
            SERPAPI_KEY=SERPAPI_KEY
        )
        log(f"[{job_id}] M3 OK")

        # ── Módulo 4 ──────────────────────────────────
        log(f"[{job_id}] M4 — generando lookbook")
        ruta_jpg, ruta_html = modulo4_generar_imagen(
            resultado_mod3=resultado_m3,
            armario=armario,
            carpeta_salida=CARPETA_SALIDA,
            model=gemini_model
        )
        log(f"[{job_id}] M4 OK → {ruta_html}")

        # ── Leer HTML ─────────────────────────────────
        with open(ruta_html, "r", encoding="utf-8") as f:
            html_content = f.read()

        # Incrustar imágenes en base64 para que el iframe las muestre
        import re
        def img_a_b64(match):
            nombre = match.group(1)
            ruta_img = os.path.join(CARPETA_SALIDA, nombre)
            if os.path.exists(ruta_img):
                with open(ruta_img, "rb") as img_f:
                    b64 = base64.b64encode(img_f.read()).decode()
                return f'src="data:image/jpeg;base64,{b64}"'
            return match.group(0)
        html_content = re.sub(r'src="([^"]+\.jpg)"', img_a_b64, html_content)

        # ── Metadata ──────────────────────────────────
        outfits = resultado_m3.get("outfits", [resultado_m3.get("outfit", {})])
        primera = outfits[0] if outfits else {}
        ctx     = resultado_m1.get("contexto", {})

        with _lock:
            _resultado_pendiente[job_id] = {
                "ok": True,
                "html_content": html_content,
                "ocasion":      primera.get("ocasion", ""),
                "temperatura":  ctx.get("temperatura"),
                "ciudad":       ctx.get("ciudad", ciudad),
                "num_prendas":  sum(len(o.get("prendas", [])) for o in outfits),
            }
        log(f"[{job_id}] ✅ Completado")

    except Exception as e:
        log(f"[{job_id}] ❌ ERROR: {e}")
        log(traceback.format_exc())
        with _lock:
            _resultado_pendiente[job_id] = {"ok": False, "error": str(e)}


# ─────────────────────────────────────────────
# ENDPOINT OUTFIT — acepta la petición y espera
# ─────────────────────────────────────────────
@app.route("/outfit", methods=["POST", "OPTIONS"])
def generar_outfit():
    if request.method == "OPTIONS":
        return jsonify({"ok": True})
        response.headers["Access-Control-Allow-Origin"] = "*"
        response.headers["Access-Control-Allow-Headers"] = "Content-Type,Authorization,ngrok-skip-browser-warning"
        response.headers["Access-Control-Allow-Methods"] = "GET,POST,OPTIONS"
        return response, 200

    instruccion = request.form.get("instruccion", "").strip()
    ciudad      = request.form.get("ciudad", "").strip() or None
    audio_file  = request.files.get("audio")
    fotos       = request.files.getlist("fotos")

    # Guardar fotos
    fotos_paths = []
    for foto in fotos:
        if foto and foto.filename:
            ruta = os.path.join(CARPETA_ROPA, foto.filename)
            foto.save(ruta)
            fotos_paths.append(ruta)

    # Guardar audio si existe
    ruta_audio = None
    if audio_file:
        with tempfile.NamedTemporaryFile(suffix=".webm", delete=False) as tmp:
            audio_file.save(tmp.name)
            ruta_audio = tmp.name

    if not instruccion and not ruta_audio:
        return jsonify({"error": "Se necesita texto o audio"}), 400

    # Crear job ID único
    job_id = str(int(time.time() * 1000))
    with _lock:
        _resultado_pendiente[job_id] = None

    # Encolar en el hilo principal usando un evento
    evento = threading.Event()

    def tarea():
        _procesar_outfit(job_id, instruccion, ciudad, ruta_audio, fotos_paths)
        evento.set()

    # Ejecutar en hilo nuevo pero con join para esperar resultado
    t = threading.Thread(target=tarea)
    t.start()

    # Esperar hasta 5 minutos
    completado = evento.wait(timeout=600)

    if not completado:
        return jsonify({"error": "Timeout: el procesamiento tardó más de 5 minutos"}), 504

    with _lock:
        resultado = _resultado_pendiente.pop(job_id, None)

    if not resultado:
        return jsonify({"error": "Sin resultado"}), 500

    if not resultado.get("ok"):
        return jsonify({"error": resultado.get("error", "Error desconocido")}), 500

    resultado.pop("ok")
    return jsonify(resultado)


# ─────────────────────────────────────────────
# ARRANQUE
# ─────────────────────────────────────────────
ngrok.set_auth_token(NGROK_TOKEN)

# Tunnel en hilo secundario
def run_tunnel():
    time.sleep(3)
    tunnel = ngrok.connect(5000)
    log(f"✅ URL pública: {tunnel.public_url}")

threading.Thread(target=run_tunnel, daemon=True).start()

# Flask en el hilo PRINCIPAL — así Gemini no tiene problemas de proxy
log("Arrancando Flask en hilo principal...")
app.run(port=5000, use_reloader=False, debug=False)

print("─" * 50)
print(f"✅  URL pública: {tunnel.public_url}")
print(f"    Pégala en el HTML como URL del servidor")
print("─" * 50)
